# 05 — Commission decision regex in Court documents

This notebook searches processed InfoCuria Court documents for **references to European Commission decisions**.

The aim is deliberately narrow. It does **not** search for other Court case references, legislation, or general legal citations. It looks for three Commission identifier families:

- `decision_case_number` — Commission competition case/procedure numbers such as `IV/61`, `AT.39904`, `COMP/37.792`, `COMP/F-2/32432`, or `COMP/F2/99912`;
- `decision_oj_reference` — published Commission decision references with an explicit suffix, such as `233/64/EEC`, `98/531/EC`, or `2004/337/EC`;
- `decision_celex` — explicitly printed CELEX decision identifiers such as `31964D0233`.

The regex step is a **candidate-generation step**. A matching string is retained even when the words `Commission` or `Decision` are not nearby. For each unique normalized reference within a Court document, the output records:

- how often the reference occurs;
- how many occurrences have `Commission` within the context window;
- how many occurrences have `Decision` within the context window; and
- how many occurrences have both nearby.

No CELEX identifier is inferred from another reference.

## Inputs and outputs

Input: `output/court_infocuria/court_clean_file_manifest.xlsx`, produced by `04_process_court_documents`.

Only rows with `processing_status` equal to `success` or `review` and a usable `clean_file_path` are scanned.

Output:

- `output/court_regex/court_decision_regex.xlsx`
- `logs/court_regex/court_decision_regex_run_info.json`

The Excel file contains one row per **unique normalized Commission reference per Court document**.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import os
import re

import pandas as pd
from tqdm.auto import tqdm

## 1. Paths and settings

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    override = os.environ.get("ECCJEU_PROJECT_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"ECCJEU_PROJECT_ROOT does not exist: {root}")
        return root

    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "code" / "03_matching").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ec_cjeu project root. Run the notebook from within "
        "the repository or set ECCJEU_PROJECT_ROOT."
    )


PROJECT_ROOT = find_project_root()

INPUT_MANIFEST = (
    PROJECT_ROOT
    / "output"
    / "court_infocuria"
    / "court_clean_file_manifest.xlsx"
)

OUTPUT_DIR = PROJECT_ROOT / "output" / "court_regex"
LOG_DIR = PROJECT_ROOT / "logs" / "court_regex"

OUTPUT_XLSX = OUTPUT_DIR / "court_decision_regex.xlsx"
RUN_INFO_JSON = LOG_DIR / "court_decision_regex_run_info.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Characters taken on each side of every occurrence when checking
# whether the words "Commission" and/or "Decision" are nearby.
CONTEXT_CHARS = 250


def repo_path(path: Path | str) -> str:
    path = Path(path).resolve()
    try:
        return path.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def resolve_repo_path(value: Any) -> Path | None:
    text = str(value or "").strip()
    if not text:
        return None

    path = Path(text)

    if path.is_absolute():
        if path.exists():
            return path.resolve()

        # Tolerate old absolute paths containing the repository name.
        parts = list(path.parts)
        for repo_name in ("ec_cjeu", "eccjeu"):
            if repo_name in parts:
                idx = parts.index(repo_name)
                rel = Path(*parts[idx + 1:])
                candidate = PROJECT_ROOT / rel
                if candidate.exists():
                    return candidate.resolve()

        return path.resolve()

    return (PROJECT_ROOT / path).resolve()


print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_MANIFEST)
print("Output:", OUTPUT_XLSX)
print("Context window:", f"±{CONTEXT_CHARS} characters")

Project root: /home/edik/projects/ec_cjeu
Input: /home/edik/projects/ec_cjeu/output/court_infocuria/court_clean_file_manifest.xlsx
Output: /home/edik/projects/ec_cjeu/output/court_regex/court_decision_regex.xlsx
Context window: ±250 characters


## 2. Load the processed Court-document manifest

Notebook 04 has already linked Court metadata to the clean UTF-8 text files. This notebook therefore starts from the clean-file manifest rather than from the raw download manifest.

In [3]:
if not INPUT_MANIFEST.exists():
    raise FileNotFoundError(f"Required clean-file manifest is missing: {INPUT_MANIFEST}")

manifest = pd.read_excel(INPUT_MANIFEST, dtype=str, keep_default_na=False)
manifest.columns = [str(c).strip() for c in manifest.columns]

required_columns = {
    "record_id",
    "affId",
    "procedureId",
    "document_type",
    "clean_file_path",
    "processing_status",
}

missing = sorted(required_columns - set(manifest.columns))
if missing:
    raise ValueError(f"Clean-file manifest is missing required columns: {missing}")

work = manifest.loc[
    manifest["processing_status"].str.strip().str.lower().isin({"success", "review"})
    & manifest["clean_file_path"].str.strip().ne("")
].copy()

work["resolved_text_path"] = work["clean_file_path"].map(resolve_repo_path)
work["text_file_exists"] = work["resolved_text_path"].map(
    lambda p: bool(p is not None and p.exists() and p.is_file())
)

missing_text = work.loc[~work["text_file_exists"]].copy()
work = work.loc[work["text_file_exists"]].reset_index(drop=True)

print(f"Rows in clean-file manifest: {len(manifest):,}")
print(f"Usable processed Court documents: {len(work):,}")
print(f"Missing referenced text files: {len(missing_text):,}")

display(
    work[
        [
            "record_id",
            "affId",
            "procedureId",
            "document_type",
            "processing_status",
            "clean_file_path",
        ]
    ].head(20)
)

Rows in clean-file manifest: 42,847
Usable processed Court documents: 42,847
Missing referenced text files: 0


,record_id,affId,procedureId,document_type,processing_status,clean_file_path
0,id_85906__000001,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/pdf_85906-...
1,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_46950...
2,id_49061__000003,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_49061...
3,id_55247__000004,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_55247...
4,id_65336__000005,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_65336...
5,id_49453__000006,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_49453...
6,id_57812__000007,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_57812...
7,id_64087__000008,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_64087...
8,id_61266__000009,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET_SOM,success,data/processed/court_infocuria/text/html_61266...
9,id_64718__000010,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET,success,data/processed/court_infocuria/text/html_64718...


## 3. Commission-reference regexes

The patterns are intentionally restricted to Commission-related identifiers.

### Commission competition case numbers

Simple forms include `IV/61`, `IV/26.760`, `COMP/37.792`, and `AT.39904`. At least two digits are required, so a string such as `IV/4` is not accepted.

A second pattern captures COMP sub-series such as `COMP/F-2/32432` and `COMP/F2/99912`.

### Published Commission decision references

Only references with an explicit suffix are retained, for example `233/64/EEC`, `98/531/EC`, or `2004/337/EC`. Suffix-free references are ignored.

### Commission document identifiers

The pattern requires `C`, optional whitespace, a four-digit year in **round parentheses**, a 1–5 digit number, and an optional `final` suffix.

### CELEX

The CELEX pattern is deliberately strict: `3` + four-digit year + `D` + exactly four digits, with no internal spaces.

In [4]:
# Simple Commission case numbers:
# IV/61, IV/26.760, COMP/37.792, AT.39904, etc.
COMMISSION_CASE_SIMPLE_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (?P<prefix>COMP|IV|AT)
    \s*
    [./\\-]
    \s*
    (?P<number>
        \d{2,6}
        |
        \d{1,2}\.\d{3,5}(?:\.\d+)?
    )
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

# COMP sub-series:
# COMP/F-2/32432, COMP/F2/99912
COMMISSION_CASE_SUBSERIES_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    COMP
    \s*/\s*
    (?P<section>[A-Z](?:\s*-\s*)?\d?)
    \s*/\s*
    (?P<number>\d{2,6})
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

# Published Commission decision / OJ references.
# Only suffix-bearing forms are retained.
DECISION_OJ_REFERENCE_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    (?P<part1>\d{1,4})
    \s*/\s*
    (?P<part2>\d{1,4})
    \s*/\s*
    (?P<suffix>EEC|EC|EU|ECSC|EURATOM)
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)


# Explicit CELEX decision identifiers only.
CELEX_RE = re.compile(
    r"""
    (?<![A-Za-z0-9])
    3
    (?P<year>\d{4})
    D
    (?P<number>\d{4})
    (?![A-Za-z0-9])
    """,
    re.IGNORECASE | re.VERBOSE,
)

COMMISSION_WORD_RE = re.compile(r"\bCommission\b", re.IGNORECASE)
DECISION_WORD_RE = re.compile(r"\bDecision\b", re.IGNORECASE)


def normalize_case_simple(match: re.Match) -> str:
    prefix = match.group("prefix").upper()
    number = re.sub(r"\s+", "", match.group("number"))

    if prefix == "AT":
        return f"AT.{number}"
    return f"{prefix}/{number}"


def normalize_case_subseries(match: re.Match) -> str:
    section = re.sub(r"\s+", "", match.group("section")).upper()
    number = match.group("number")
    return f"COMP/{section}/{number}"


def normalize_oj_reference(match: re.Match) -> str:
    return (
        f"{match.group('part1')}/"
        f"{match.group('part2')}/"
        f"{match.group('suffix').upper()}"
    )



def normalize_celex(match: re.Match) -> str:
    return f"3{match.group('year')}D{match.group('number')}"


REGEX_FAMILIES = [
    (
        "decision_case_number",
        COMMISSION_CASE_SUBSERIES_RE,
        normalize_case_subseries,
    ),
    (
        "decision_case_number",
        COMMISSION_CASE_SIMPLE_RE,
        normalize_case_simple,
    ),
    (
        "decision_oj_reference",
        DECISION_OJ_REFERENCE_RE,
        normalize_oj_reference,
    ),
    (
        "decision_celex",
        CELEX_RE,
        normalize_celex,
    ),
]

## 4. Scan and aggregate references

Every regex occurrence is checked in a fixed character window around the match. The words `Commission` and `Decision` are counted independently.

Repeated appearances of the same normalized reference in the same Court document are then collapsed to one row. The output therefore describes both the reference and how consistently it appears in Commission/decision context.

In [5]:
def collapse_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def occurrence_context(text: str, start: int, end: int) -> str:
    left = max(0, start - CONTEXT_CHARS)
    right = min(len(text), end + CONTEXT_CHARS)
    return text[left:right]


def scan_text(text: str) -> list[dict[str, Any]]:
    occurrences: list[dict[str, Any]] = []

    for reference_type, pattern, normalizer in REGEX_FAMILIES:
        for match in pattern.finditer(text):
            context = occurrence_context(text, match.start(), match.end())

            commission_nearby = bool(COMMISSION_WORD_RE.search(context))
            decision_nearby = bool(DECISION_WORD_RE.search(context))

            occurrences.append(
                {
                    "reference_type": reference_type,
                    "matched_text": match.group(0),
                    "reference_normalized": normalizer(match),
                    "start_char": match.start(),
                    "end_char": match.end(),
                    "commission_nearby": commission_nearby,
                    "decision_nearby": decision_nearby,
                    "commission_and_decision_nearby": (
                        commission_nearby and decision_nearby
                    ),
                    "context": collapse_whitespace(context),
                }
            )

    return occurrences


# Cache by physical clean-text path so duplicate manifest rows do not
# cause the same text file to be read and scanned repeatedly.
scan_cache: dict[str, list[dict[str, Any]]] = {}

rows: list[dict[str, Any]] = []

for _, doc in tqdm(
    work.iterrows(),
    total=len(work),
    desc="Scanning Court documents for Commission references",
):
    text_path = Path(doc["resolved_text_path"])
    cache_key = str(text_path.resolve())

    if cache_key not in scan_cache:
        text = text_path.read_text(encoding="utf-8", errors="replace")
        scan_cache[cache_key] = scan_text(text)

    occurrences = scan_cache[cache_key]
    if not occurrences:
        continue

    occ_df = pd.DataFrame(occurrences)

    for (reference_type, reference_normalized), group in occ_df.groupby(
        ["reference_type", "reference_normalized"],
        sort=False,
        dropna=False,
    ):
        group = group.sort_values(["start_char", "end_char"])
        first = group.iloc[0]

        matched_forms = list(dict.fromkeys(group["matched_text"].astype(str)))

        rows.append(
            {
                "record_id": doc.get("record_id", ""),
                "affId": doc.get("affId", ""),
                "procedureId": doc.get("procedureId", ""),
                "document_type": doc.get("document_type", ""),
                "document_date": doc.get("document_date", ""),
                "case_number_clean": doc.get("case_number_clean", ""),
                "clean_file_path": doc.get("clean_file_path", ""),
                "reference_type": reference_type,
                "reference": " | ".join(matched_forms),
                "reference_normalized": reference_normalized,
                "reference_count": int(len(group)),
                "commission_nearby_count": int(group["commission_nearby"].sum()),
                "decision_nearby_count": int(group["decision_nearby"].sum()),
                "commission_and_decision_nearby_count": int(
                    group["commission_and_decision_nearby"].sum()
                ),
                "context": first["context"],
            }
        )


OUTPUT_COLUMNS = [
    "record_id",
    "affId",
    "procedureId",
    "document_type",
    "document_date",
    "case_number_clean",
    "clean_file_path",
    "reference_type",
    "reference",
    "reference_normalized",
    "reference_count",
    "commission_nearby_count",
    "decision_nearby_count",
    "commission_and_decision_nearby_count",
    "context",
]

results = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)

if len(results):
    results = (
        results.sort_values(
            [
                "affId",
                "procedureId",
                "record_id",
                "reference_type",
                "reference_normalized",
            ],
            kind="stable",
        )
        .reset_index(drop=True)
    )

print(f"Documents scanned: {len(work):,}")
print(
    "Documents with at least one Commission-reference candidate:",
    f"{results['record_id'].nunique() if len(results) else 0:,}",
)
print(f"Unique document-reference rows: {len(results):,}")

if len(results):
    display(
        results["reference_type"]
        .value_counts()
        .rename_axis("reference_type")
        .reset_index(name="rows")
    )

display(results.head(30))

Scanning Court documents for Commission references:   0%|          | 0/42847 [00:00<?, ?it/s]

Documents scanned: 42,847
Documents with at least one Commission-reference candidate: 16,497
Unique document-reference rows: 38,999


,reference_type,rows
0,decision_oj_reference,37058
1,decision_case_number,1941


,record_id,affId,procedureId,document_type,document_date,case_number_clean,clean_file_path,reference_type,reference,reference_normalized,reference_count,commission_nearby_count,decision_nearby_count,commission_and_decision_nearby_count,context
0,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,1999/514/EC,1999/514/EC,3,3,3,3,Decision 94/474/EC and repealing Decision 96/2...
1,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,64/433/EEC,64/433/EEC,1,1,1,1,"t of the experience gained, the Commission wil..."
2,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,77/99/EEC,77/99/EEC,1,1,1,1,will endeavour to clarify and complete if need...
3,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,89/662/EEC,89/662/EEC,4,2,1,1,7 December 1992 laying down animal health and ...
4,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,90/425/EEC,90/425/EEC,1,0,1,0,porting from its territory to the other Member...
5,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,92/118/EEC,92/118/EEC,1,0,0,0,of 26 June 1990 concerning veterinary and zoot...
6,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,94/474/EC,94/474/EC,3,3,3,3,"PLICATION for a declaration that, by refusing ..."
7,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,96/239/EC,96/239/EC,4,4,4,4,by refusing to adopt the measures necessary in...
8,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,97/404/EC,97/404/EC,1,1,1,1,orwarded to the Commission the opinion issued ...
9,id_46950__000002,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET,2001-12-13,C-1/00,data/processed/court_infocuria/text/html_46950...,decision_oj_reference,98/256/EC,98/256/EC,4,4,4,4,"lle, and then by R. Loosli-Surrans and G. de B..."


## 5. Save the Court Commission-reference table

In [6]:
# Excel's XML format rejects a small set of ASCII control characters that can
# appear in scraped/OCR'd court text. Sanitize only the export copy so that
# the underlying `results` dataframe remains unchanged for later analysis.
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

results_excel = results.copy()
for col in results_excel.select_dtypes(include=["object"]).columns:
    results_excel[col] = results_excel[col].map(
        lambda value: ILLEGAL_CHARACTERS_RE.sub("", value)
        if isinstance(value, str)
        else value
    )

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    results_excel.to_excel(writer, sheet_name="references", index=False)

    ws = writer.sheets["references"]
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    # Keep the review table readable without elaborate formatting.
    widths = {
        "A": 22,
        "B": 18,
        "C": 18,
        "D": 16,
        "E": 14,
        "F": 18,
        "G": 48,
        "H": 26,
        "I": 28,
        "J": 28,
        "K": 16,
        "L": 24,
        "M": 22,
        "N": 36,
        "O": 90,
    }
    for column, width in widths.items():
        ws.column_dimensions[column].width = width


run_info = {
    "created_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "input_manifest": repo_path(INPUT_MANIFEST),
    "output_file": repo_path(OUTPUT_XLSX),
    "context_chars_each_side": CONTEXT_CHARS,
    "eligible_manifest_rows": int(len(work)),
    "missing_clean_text_files": int(len(missing_text)),
    "documents_with_candidates": int(
        results["record_id"].nunique() if len(results) else 0
    ),
    "unique_document_reference_rows": int(len(results)),
}

RUN_INFO_JSON.write_text(
    json.dumps(run_info, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved:", repo_path(OUTPUT_XLSX))
print("Run log:", repo_path(RUN_INFO_JSON))

Saved: output/court_regex/court_decision_regex.xlsx
Run log: logs/court_regex/court_decision_regex_run_info.json
